In [1]:
import pickle

# run the model
with open('../xgb_model_withgpa.pkl', 'rb') as file:
    xgb_model = pickle.load(file)



In [2]:
import pandas as pd
import numpy as np

In [3]:
xgb_input = pd.read_csv("../x.csv")
xgb_input.head(10)

,Unnamed: 0,anxious,calm,conventional,critical,dependable,disorganized,enthusiastic,experiences,reserved,...,hour,exercise,walk,have,schedule,sleep_hours,sleep_quality,sleepiness,number,gpa_all
0,0,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
1,1,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
2,2,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
3,3,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
4,4,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
5,5,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
6,6,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
7,7,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
8,8,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505
9,9,1.0,5.0,2.0,2.0,2.0,3.0,5.0,2.0,2.0,...,4.0,3.0,2.0,1.0,1.0,8.0,2.0,1.0,3.0,3.505


## Noise experiment on highly relevent variables

In [4]:
print("sadornot distinct values:", xgb_input['sadornot'].unique())
print("experiences distinct values:", xgb_input['experiences'].unique())
print("gpa_all distinct values:", xgb_input['gpa_all'].unique())


sadornot distinct values: [1. 2.]
experiences distinct values: [2. 4. 3. 1. 5.]
gpa_all distinct values: [3.505 3.029 3.474 3.705 3.667 3.245 3.293 3.373 3.476 3.947 3.719 3.826
 2.815 3.79  3.625 2.4   3.519]


In [5]:
# add noise to variables: sadornot
# Randomly add 1, subtract 1, or make no change for each sample.

import numpy as np

noisy_input = xgb_input.copy()

sadornot_noise = np.random.choice([-1, 0, 1], size=noisy_input.shape[0])

noisy_input['sadornot'] = noisy_input['sadornot'] + sadornot_noise
noisy_input['sadornot'] = noisy_input['sadornot'].clip(lower=1, upper=2)

print(noisy_input['sadornot'].value_counts().sort_index())



sadornot
1.0    4643
2.0    4360
Name: count, dtype: int64


In [16]:
# add noise to variable: experiences
# apply normal distribution to choose the reasonable noise to add on the original experiences column

experiences_noise = np.random.choice([-1, 0, 1], size=noisy_input.shape[0], p=[0.25, 0.5, 0.25])

noisy_input['experiences'] = noisy_input['experiences'] + experiences_noise
noisy_input['experiences'] = noisy_input['experiences'].clip(lower=1, upper=5)

print(noisy_input['experiences'].head(10))



0    3.0
1    2.0
2    3.0
3    3.0
4    3.0
5    2.0
6    3.0
7    1.0
8    3.0
9    3.0
Name: experiences, dtype: float64


In [7]:
# add noise to variable: gpa_all
# gpa_all is continuous numerical variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

gpa_noise = np.random.normal(0, 0.05, size=noisy_input.shape[0])
noisy_input['gpa_all'] = noisy_input['gpa_all'] + gpa_noise
print(noisy_input['gpa_all'].head(10))

0    3.450172
1    3.448987
2    3.544571
3    3.472069
4    3.543474
5    3.443424
6    3.417603
7    3.529870
8    3.442150
9    3.540710
Name: gpa_all, dtype: float64


In [8]:
# re-run the xgboost model
noisy_input = noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

y_pred_noisy = xgb_model.predict(noisy_input)


In [9]:
from sklearn.metrics import mean_squared_error
y_data = pd.read_csv("../y.csv")

y_true = y_data['total_score']
mse_noisy = mean_squared_error(y_true, y_pred_noisy)

print("MSE after adding noise on highly relevant variables:", mse_noisy)


MSE after adding noise on highly relevant variables: 16.962799072265625


In [10]:
from sklearn.metrics import r2_score
r2_noisy = r2_score(y_true, y_pred_noisy)
print("R² after adding noise on highly relevant variables:", r2_noisy)

R² after adding noise on highly relevant variables: 0.4685271382331848


## Noise experiment on less relevent variables

In [11]:
print("has_negative_text distinct values:", xgb_input['has_negative_text'].unique())
print("schedule distinct values:", xgb_input['schedule'].unique())
print("have distinct values:", xgb_input['have'].unique())

has_negative_text distinct values: [1. 0.]
schedule distinct values: [1. 2.]
have distinct values: [1. 2.]


In [12]:
import numpy as np

less_relevent_noisy_input = xgb_input.copy()

binary_features = {
    'has_negative_text': (0, 1),
    'schedule': (1, 2),
    'have': (1, 2)
}

for feature, (min_val, max_val) in binary_features.items():
    noise = np.random.choice([-1, 0, 1], size=less_relevent_noisy_input.shape[0])
    less_relevent_noisy_input[feature] = less_relevent_noisy_input[feature] + noise
    less_relevent_noisy_input[feature] = less_relevent_noisy_input[feature].clip(lower=min_val, upper=max_val)

print(less_relevent_noisy_input[list(binary_features.keys())].head(10))


   has_negative_text  schedule  have
0                0.0       1.0   2.0
1                0.0       1.0   1.0
2                0.0       1.0   1.0
3                1.0       1.0   1.0
4                0.0       2.0   1.0
5                1.0       1.0   1.0
6                1.0       1.0   2.0
7                0.0       1.0   2.0
8                1.0       1.0   1.0
9                0.0       2.0   2.0


In [13]:
# re-run the xgboost model

less_relevent_noisy_input = less_relevent_noisy_input.drop(columns=['Unnamed: 0'], errors='ignore')

less_relevent_y_pred_noisy = xgb_model.predict(less_relevent_noisy_input)



In [14]:
less_relevent_mse_noisy = mean_squared_error(y_true, less_relevent_y_pred_noisy)

print("MSE after adding noise for less relevant variables:", less_relevent_mse_noisy)

MSE after adding noise for less relevant variables: 3.2133333683013916


In [15]:
less_relevent_r2_noisy = r2_score(y_true, less_relevent_y_pred_noisy)
print("R² after adding noise on less relevant variables:", less_relevent_r2_noisy)

R² after adding noise on less relevant variables: 0.8993209004402161


| Condition (XGBoost Model)                         | MSE       | RMSE (√MSE) | R²      |
|:----------------------------------|:----------|:------------|:--------|
| No Noise                          | 3.3773    | 1.8374       | 0.8963  |
| Noise on Top-3 Important Features | 16.9628   | 4.1186       | 0.4685  |
| Noise on Bottom-3 Important Features   | 3.2133    | 1.7915       | 0.8993  |

